# Python. Семинар 4

In [15]:
class HashableList:
    def __init__(self, data):
        self.data = data

    # Обманываем Python: возвращаем хэш от неизменяемого кортежа
    def __hash__(self):
        return hash(tuple(self.data))

    # Сравниваем внутренние данные
    def __eq__(self, other):
        return self.data == other.data

    def __repr__(self):
        return str(self.data)


# Создаем список и оборачиваем его
my_list = [1, 2, 3]
fooled_key = HashableList(my_list)

# Используем как ключ словаря
my_dict = {fooled_key: "Успех!"}
print(my_dict)  # Выведет: {[1, 2, 3]: 'Успех!'}

# ВНИМАНИЕ: Если мы изменим список внутри, хэш сломается!
my_list.append(4)

{[1, 2, 3]: 'Успех!'}


---

## Часть 1. Быстрый recap лекции

### 1.1. Переменные — стикеры, а не коробки

Присваивание не кладёт значение в именованную коробку. Оно приклеивает к уже
созданному объекту ещё один стикер с именем. Подтверждение — одна строка:

In [ ]:
a = [1, 2, 3]
b = a                # не копия, а второй стикер на тот же объект
b.append(4)

a, a is b

### 1.2. `is` против `==`

`is` спрашивает «тот же ли это объект» (сравнивает `id`), `==` — «равно ли
содержимое» (вызывает `__eq__`). Импостер из лекции: равен, но другой.

In [ ]:
cookie = {"name": "Cookie", "size": "small"}
brownie = cookie                                 # тот же объект
imposter = {"name": "Cookie", "size": "small"}   # другой, но такой же

print(cookie is brownie, cookie == imposter, cookie is imposter)
print(id(cookie) == id(brownie), id(cookie) == id(imposter))

### 1.3. Список под капотом

Список хранит не объекты, а **ссылки** на них — по 8 байт каждая (внутри
CPython ссылка — это C-указатель `PyObject*`). Поэтому элементы могут быть
разных типов, а доступ по индексу — арифметика адресов: `head + idx * 8`,
сложность O(1). Место под ссылки берётся с запасом и расширяется по мере
роста — это амортизация: `append` стоит O(1) в среднем, а не в каждом вызове.

In [ ]:
guests = ["Frank", "Claire", "Zoe", True, 42]   # типы разные — ссылки на что угодно

guests[3], type(guests[3]).__name__

### 1.4. Относительная неизменяемость

Кортеж не даёт заменить свои элементы — но не следит за тем, что происходит
внутри них. Ссылки неприкосновенны, содержимое — нет.

In [16]:
t = (1, 2, [3, 4])

t[-1].append(5)     # можно: меняли список, а не кортеж
print(t)
try:
    t[0] = 0        # нельзя: элемент кортежа не переприсваивается
except TypeError as e:
    print("TypeError:", e)

(1, 2, [3, 4, 5])
TypeError: 'tuple' object does not support item assignment


### 1.5. Копии: поверхностная и глубокая

`list(l)`, `l[:]` и `l.copy()` дублируют внешний контейнер, вложения остаются
общими. `copy.deepcopy` копирует всё до дна. Поверхностной достаточно, когда
внутри неизменяемое; изменяемые вложения — повод для глубокой копии.

In [17]:
from copy import deepcopy

l1 = [1, [2, 3]]
l2 = l1.copy()     # поверхностная
l3 = deepcopy(l1)    # глубокая

l1[1].append(99)
print(l1, l2, l3)
print(l2[1] is l1[1], l3[1] is l1[1])

[1, [2, 3, 99]] [1, [2, 3, 99]] [1, [2, 3]]
True False


### 1.6. `memoryview`: окно вместо копии

Срез `bytes`/`bytearray`/`array` — это копия. `memoryview` не копирует данные,
а смотрит на чужой буфер: размер окна фиксирован и от длины не зависит,
а содержимое живёт вместе с буфером.

In [18]:
data = bytearray(b"abc")
view = memoryview(data)

data[0] = ord("z")   # изменили буфер
print(bytes(view))       # окно увидело: копия осталась бы в старом состоянии
view[1] = ord('c')   
print(bytes(view))

b'zbc'
b'zcc'


### 1.7. Функции

Функция — тоже объект: у неё есть имя, докстринг, аннотации; её можно присвоить
переменной, передать аргументом, положить в словарь. Без `return` функция
возвращает `None`.

In [ ]:
def is_even(n: int) -> bool:
    """Проверяет число на чётность."""
    return n % 2 == 0

print(is_even(3), type(is_even).__name__)
print(is_even.__doc__, is_even.__annotations__)

f = is_even          # второй стикер на объект-функцию
f(4)

### 1.8. Рекурсия

Базовый случай останавливает спуск, рекурсивный шаг приближает к нему. Оба
обязаны быть — иначе спуск не кончится.

In [ ]:
def factorial(n: int) -> int:
    """returns n!"""
    return 1 if n < 2 else n * factorial(n - 1)

factorial(5)

### 1.9. Функции высшего порядка и `lambda`

Функция высшего порядка принимает функцию аргументом или возвращает функцию:
`sorted(..., key=len)`, `map`, `filter`. `lambda` — безымянная функция одной
строки; гайд из лекции: если лямбде нужен комментарий — ей нужно имя и `def`.

In [ ]:
fruits = ["strawberry", "fig", "apple", "cherry", "raspberry", "banana"]

print(sorted(fruits, key=len))
print(sorted(fruits, key=lambda word: word[::-1]))
print([w for w in fruits if len(w) <= 5])

---

## Часть 2. Углубление

### 2.0. Перепривязка против мутации

Всё, что семинар будет делать дальше, стоит на одном различии. **Мутация** меняет
сам объект — изменение видно через каждое имя, которое на него смотрит.
**Перепривязка** (`name = ...`) отцепляет одно имя и прицепляет его к другому
объекту — остальные имена остаются у старого.

In [10]:
values = [1, 2]
alias = values

values.append(3)          # мутация: объект тот же, содержимое выросло
print(alias)

values = [9]              # перепривязка: имя переехало, alias остался
print(alias, values is alias)

[1, 2, 3]
[1, 2, 3] False


`+=` для списка — мутация (это `extend`), `a = a + [...]` — перепривязка;
присваивание срезу `a[:] = ...` — мутация без смены объекта. Это разделы 2 и 3
семинара. Якорь: `inplace.py`.

### 2.1. Ссылка, `id` и указатель — три разных вещи

**Ссылка** — понятие языка Python: имя или ячейка контейнера, привязанная
к объекту. Присваивание копирует ссылку, `is` их сравнивает. **Указатель** —
понятие C: адрес, которым управляет сам программист — сдвинуть, разименовать,
освободить память. Таких операций в Python нет: CPython хранит ссылку как
C-указатель `PyObject*`, но он спрятан внутри интерпретатора.

`id()` — самое близкое к адресу, что язык выдаёт наружу. В CPython это адрес
объекта, но в виде **непрозрачного числа**: по нему нельзя получить объект
обратно, нельзя сдвинуть «к соседу» и прочитать память, нельзя освободить.
Гарантии языка ровно две: пока объект жив, его `id` не меняется; у двух живых
объектов `id` разные. Адрес это или просто номер — деталь реализации: в PyPy
`id` адресом не является.

In [19]:
x = [1, 2, 3]
print(hex(id(x)))       # CPython: выглядит как адрес — но это обычное число

neighbour = id(x) + 8    # сдвинуть число можно
print(neighbour)         # а вот «прочитать по адресу» из языка нельзя

0x7476cd50c880
128053599586440


Из `id` нельзя вернуться к объекту. Единственная осмысленная операция с ним —
сравнение на равенство, и она уже встроена в язык:

In [ ]:
a = [1]
b = a                    # скопировалась ссылка — аналог копии указателя в C

print(id(a) == id(b), a is b)

### 2.2. `id` уникален только среди живых

`id` — номер объекта. Номер занят, пока жив объект: как только он собран,
тот же адрес может достаться новому. Поэтому сравнивать сохранённые
на будущее `id` нельзя — только `a is b`, пока оба живы.

In [20]:
print(id([]) == id([]))        # True: первый список уже умер, адрес освободился

first, second = [], []         # живут одновременно — адреса разные
print(id(first) == id(second), first is second)

True
False False


Якорь: семинар, раздел 6; ДЗ `aliases`.

### 2.3. Почему `is` на числах и строках врёт

CPython заранее создаёт маленькие числа (от −5 до 256) в единственном экземпляре
и делит одинаковые строковые литералы. Поэтому `is` на них часто отвечает
`True` — и закрепляет антипаттерн. Значение, рождённое вычислением в рантайме, —
свежий объект.

In [ ]:
a, b = 5, 5
print(a is b)               # True: маленькие числа из кэша

n = 300
c, d = n + n, n + n         # считаются в рантайме: каждая сумма — новый объект
print(c == d, c is d)

In [ ]:
word = "hello"
literal = "hello"
computed = "".join(("hel", "lo"))   # строка собрана в рантайме

print(word is literal, computed == word, computed is word)

Вывод: `is` уместен только там, где вопрос действительно про личность объекта —
`value is None`. Якорь: ДЗ `aliases`, функция `describe`.

### 2.4. Одна строка на всю матрицу

Умножение списка повторяет ссылки, а не содержимое. Внутренний `[0] * 3`
безопасен: нули неизменяемы, сколько ни дели ссылку. Опасно умножать изменяемое.

In [21]:
grid = [[0] * 3] * 3
print(grid[0] is grid[1] is grid[2])

grid[0][0] = 1
grid

True


[[1, 0, 0], [1, 0, 0], [1, 0, 0]]

In [12]:
grid = [[0] * 3 for _ in range(3)]   # тело включения выполняется на каждом шаге
print(grid[0] is grid[1])

grid[0][0] = 1
grid

False


[[1, 0, 0], [0, 0, 0], [0, 0, 0]]

### 2.5. Поверхностная копия: три записи, один смысл

`list(l)`, `l[:]` и `l.copy()` делают одно и то же: новый внешний контейнер,
внутри — ссылки на те же объекты.

In [22]:
l1 = [3, [66, 55, 44], (7, 8, 9)]
l2 = list(l1)

l1.append(100)        # рост оригинала копии не виден
l1[1].remove(55)      # мутация общего вложения — видна обоим
print("l1:", l1)
print("l2:", l2)

l1: [3, [66, 44], (7, 8, 9), 100]
l2: [3, [66, 44], (7, 8, 9)]


Такое копирование рекомендуют, когда внутри неизменяемое: ссылки на числа и
кортежи делить не страшно, зато память экономится. Якорь: `copies.py`.

### 2.6. Глубокая копия: общие ссылки и циклы

Обычное «скопировать всё в лоб» ломается в двух случаях. Во-первых, если один объект лежит в списке несколько раз, он размножится на разные копии и связь потеряется. Во-вторых, если объект ссылается сам на себя, программа уйдет в вечный цикл и зависнет. deepcopy решает обе проблемы: он ведет блокнот «уже скопированного». Если объект там есть, он просто берет готовую копию.

In [13]:
from copy import deepcopy

shared = [0]
data = [shared, shared]

copy_ = deepcopy(data)
print(copy_[0] is copy_[1], copy_[0] is shared)   # отношение сохранено, объект новый

True False


In [ ]:
a = [10, 20]
b = [a, 30]
a.append(b)                          # цикл: a → b → a

print(b[0] is a)
copy_ = deepcopy(a)
print(copy_[2] is copy_[2][0][2])    # цикл честно воссоздан внутри копии

### 2.7. Неизменяемость, которой нет

Кортеж охраняет свои ссылки, а не содержимое. Отсюда два следствия: сравнение
`==` у кортежей чувствительно к мутациям внутри, а хеша у кортежа с изменяемой
дырой нет — мостик к лекции 3: нехешируемое не годится в ключи.

In [ ]:
t1 = (1, 2, [3, 4])
t2 = (1, 2, [3, 4])
print(t1 == t2)

t1[-1].append(5)         # кортеж не помешал
print(t1, t1 == t2)      # равенство уплыло

try:
    hash(t1)             # а в ключи такой кортеж уже не годится
except TypeError as e:
    print("TypeError:", e)

True
(1, 2, [3, 4, 5]) False


TypeError: unhashable type: 'list'

Кульминация темы — `box[0] += [1]` на кортеже: операция успевает изменить список
и падает на записи результата в кортеж. Разбирается на семинаре, раздел 5.
Якорь: `boxes.py`; ДЗ `frozen` (`is_frozen`, `freeze`).

### 2.8. `memoryview`: срез без копии и его цена

Окно весит фиксированные ~170 байт независимо от длины — вместо копии,
которая стоит пропорционально размеру:

In [ ]:
from array import array

large = array("d", [i * 0.1 for i in range(1_000_000)])
view = memoryview(large)

print("оригинал:           ", large.__sizeof__())
print("срез-копия:         ", large[:100_000].__sizeof__())
print("срез-представление: ", view[:100_000].__sizeof__())

In [ ]:
data = bytearray(b"abc")
view = memoryview(data)

data[0] = ord("z")     # буфер → окно
view[1] = ord("Y")     # окно → буфер
print(data, bytes(view))

In [ ]:
data = bytearray(b"abc")
view = memoryview(data)
print(view.obj is data)    # окно знает, на чей буфер смотрит

try:
    data.append(1)         # пока буфер экспортирован, размер менять нельзя
except BufferError as e:
    print("BufferError:", e)

view.release()             # отпустить буфер можно явно
data.append(1)
print(data)

Якорь: ДЗ `window` — там окно, копия и `is_shared` через `view.obj`.

### 2.9. Сколько весит список

Пустой список — 40 байт заголовка, дальше по 8 байт на ссылку (C-указатель
`PyObject*`). Вложенные объекты не считаются: список платит только за ссылки,
сами объекты — сами.

In [ ]:
print([].__sizeof__(), [1, 2].__sizeof__(), [1, 2, []].__sizeof__() == 40 + 8 * 3)
print([1, 2, []].__sizeof__() == [1, 2, [1, 2, 1]].__sizeof__())   # ссылки одного размера

In [ ]:
steps = []
row = []
for _ in range(12):
    steps.append(row.__sizeof__())
    row.append(0)

print(steps)   # ёмкость растёт ступеньками — запас под будущие append

Запас — та самая амортизация из 1.3: место выбирается с избытком, чтобы не
платить за перебуферизацию на каждом `append`. Якорь: лекция «Список под
капотом».

### 2.10. Функция — объект с атрибутами

У объекта-функции есть `__name__`, `__doc__`, `__annotations__`. Он хешируем —
годится в ключи словаря; второе имя на нём ведёт себя как второе имя на списке.

In [ ]:
def is_even(n: int) -> bool:
    """Проверяет число на чётность."""
    return n % 2 == 0

f = is_even
print(f is is_even, f(4))

d = {is_even: "метка"}
d[is_even] = is_even      # значение — сама функция
print(d[f](6))            # f и is_even — один объект, значит один ключ

Отсюда `key=len` у `sorted` и весь функциональный стиль: функция — обычное
значение. Якорь: `sorting.py`.

### 2.11. Рекурсия на данных вместо циклов

Глубина вложенности заранее неизвестна — циклом не развернуть. Рекурсия
обходит так: число — база, список — шаг с тем же вопросом каждому элементу.

In [ ]:
def total(data):
    if isinstance(data, list):                      # база: число само себе сумма
        return sum(total(item) for item in data)    # шаг: список — спуск к каждому
    return data

total([1, [2, [3, 4]], 5])

In [ ]:
def infinite():
    infinite()

try:
    infinite()
except RecursionError:
    print("RecursionError: стек вызовов кончился")

import sys
print(sys.getrecursionlimit())

База обязана быть достижима: каждый шаг уменьшает глубину, а она конечна.
Якорь: ДЗ `nested` (total, depth, find) и `merge_deep`.

### 2.12. Мешок трюков

Правило «мутируешь на месте — возвращаешь `None`» связывает `sort`, `append`,
`reverse`. Нужен новый объект — `sorted` или срез `[::-1]`.

In [25]:
values = [3, 1, 2]
result = values.sort()          # отсортировал на месте, вернул None
print(values, result is None)

values.reverse()                # на месте, тоже None
print(values, values[::-1])     # [::-1] — новый список

[1, 2, 3] True
[3, 2, 1] [1, 2, 3]


In [ ]:
doubled = map(lambda x: x * 2, [1, 2, 3])
print(list(doubled))
print(list(doubled))                    # map ленив и одноразов

print([x * 2 for x in [1, 2, 3]])       # включение — читаемее и переиспользуемо

Якорь: `sorting.py`, `inplace.py`.

---

## Часть 3. Викторина «предскажи, что выведет»

Десять вопросов — по разделам углубления и лекции. Ответил сам → запустил
ячейку → сверился с разбором. 10 из 10 без единого запуска — к семинару готов.

### Вопрос 1

In [26]:
a = [1, 2]
b = a
b.append(3)

print(a, b, a is b)

[1, 2, 3] [1, 2, 3] True


**Разбор.** `[1, 2, 3] [1, 2, 3] True`. `b = a` — второй стикер на тот же
объект; `append` изменил объект, и неважно, через какое имя смотреть.

### Вопрос 2

In [27]:
print(id([]) == id([]))

True


**Разбор.** `True`. Первый список умер до рождения второго — освободившийся
адрес достался ему же. Для одновременно живых объектов — только `is`.

### Вопрос 3

In [28]:
a, b = 5, 5

n = 300
c, d = n + n, n + n

print(a is b, c == d, c is d)

True True False


**Разбор.** `True True False`. Пятёрки — из кэша маленьких чисел, это один
объект. Суммы считаются в рантайме: значения равны, объекты разные.

### Вопрос 4

In [29]:
grid = [[0] * 2] * 2
grid[0][0] = 9

print(grid, grid[0] is grid[1])

[[9, 0], [9, 0]] True


**Разбор.** `[[9, 0], [9, 0]] True`. Умножение повторило ссылку на одну строку:
изменили одну клетку — увидели «столбец», потому что строка-то одна и та же.
Правильная матрица: `[[0] * 2 for _ in range(2)]`.

### Вопрос 5

In [ ]:
values = [3, 1, 2]
values = values.sort()

print(values)

[1, 2, 3]


**Разбор.** `None`. Список отсортировался на месте, а `sort` вернул `None` —
его и забрало присваивание. Значение потеряно, имя затёрто.

### Вопрос 6

In [32]:
a = [1]
b = a

a += [2]
a = a + [3]

print(a, b)

[1, 2, 3] [1, 2]


**Разбор.** `[1, 2, 3] [1, 2]`. `+=` поменял общий объект — `b` увидел `[1, 2]`.
Затем `+` собрал новый список и перепривязал `a` — `b` остался у старого.

### Вопрос 7

In [33]:
box = ([],)

try:
    box[0] += [1]
except TypeError:
    print("TypeError пойман")

print(box)

TypeError пойман
([1],)


**Разбор.** `TypeError пойман`, затем `([1],)`. `box[0] += [1]` — два шага:
список изменил себя на месте (успех), результат записывается в кортеж (провал).
Операция и сработала, и упала.

### Вопрос 8

In [34]:
l1 = [1, [2]]
l2 = l1[:]

l2[0] = 99
l2[1].append(3)

print(l1, l2[0] is l1[0], l2[1] is l1[1])

[1, [2, 3]] False True


**Разбор.** `[1, [2, 3]] False True`. Поверхностная копия: замена элемента
уходит в новый внешний список, а общее вложение меняется для обоих.

### Вопрос 9

In [36]:
data = bytearray(b"abc")
view = memoryview(data)

view[0] = ord("z")

print(ord("z"))

122


**Разбор.** `bytearray(b'zbc')`. Представление пишет прямо в чужой буфер —
буфер и окно всегда видят одно и то же.

### Вопрос 10

In [ ]:
doubled = map(lambda x: x * 2, [1, 2, 3])

print(list(doubled), list(doubled))

**Разбор.** `[2, 4, 6] []`. `map` возвращает не список, а одноразовый
итератор: первый `list` его исчерпал, второму не досталось ничего.